# Práctica Pipelines - Dataset titanic

En el siguiente dataset intentaremos clasificar supervivientes del titanic en función de sus variables.

## Parte 1

Dataset: aplicando pipelines quite las columnas que considere, realice las imputaciones y/o escalamientos según corresponda y su conocimiento e intuición le indiquen. Intente clasificar usando KNN. 


## Parte 2

Reutilizando lo anterior tanto como sea posible evalue utilizando otro clasificador. Pruebe distintas alternativas de transformaciones. Evalue que opción (transformaciones+clasificador) ofrece los mejores resultados.

## Parte 3

Implemente e incorpore un transformador customizado que considere útil.

### Dataset "Titanic"


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/PabloSoligo2014/3670-UNLaM-CD/refs/heads/main/datasets/titanic.csv")

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.isnull().sum()/len(df)*100

PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64

In [6]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.neighbors import KNeighborsClassifier


def type_normalizer(X):
    return X.astype(int)


columns_to_drop = ["PassengerId", "Name", "Ticket", "Cabin"]
drop_columns_ct = ColumnTransformer(
    transformers=[
        ("drop_columns", "drop", columns_to_drop)
    ],
    verbose_feature_names_out=False,
    remainder="passthrough" ## Indicates that the remaining columns not in transformers should not be dropped from df
).set_output(transform="pandas")


categorical_columns = ["Sex", "Embarked"]
# Fill missing numeric values with the median and categories with the mode.
null_filler_ct = ColumnTransformer(
    transformers=[
        ("numeric_nulls", SimpleImputer(strategy="median"), ["Age", "Fare"]), # se podria hacer algo que me traiga los nombres de las columnas numericas con faltantes automaticamente
        ("categorical_nulls", SimpleImputer(strategy="most_frequent"), categorical_columns),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")


floats = ["Age"]
float_to_int = ColumnTransformer([("dtype", FunctionTransformer(type_normalizer, validate=False, feature_names_out="one-to-one"), floats)],
                         remainder="passthrough",
                         verbose_feature_names_out=False).set_output(transform="pandas")


numeric_columns = ["Age", "Fare", "Pclass", "SibSp", "Parch"]
scaler_ct = ColumnTransformer(
    transformers=[("numeric_scaling", StandardScaler(), numeric_columns)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")


one_hot_encoder_ct = ColumnTransformer([("OneHot", OneHotEncoder(sparse_output=False, dtype=int, drop='first') , categorical_columns)],
                         remainder='passthrough', 
                         verbose_feature_names_out=False)

pl = Pipeline([
    ("drop_columns", drop_columns_ct),
    ("null_filler", null_filler_ct),
    ("float_to_int", float_to_int),
    ("scaler", scaler_ct),
      ("one_hot_encoder", one_hot_encoder_ct),
      ("est",KNeighborsClassifier())
])

X = df.drop("Survived", axis=1)
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=3, test_size=0.3)

pl.fit(X_train, y_train)
y_pred = pl.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.79      0.79      0.79       164
           1       0.67      0.66      0.67       104

    accuracy                           0.74       268
   macro avg       0.73      0.73      0.73       268
weighted avg       0.74      0.74      0.74       268



In [11]:

from sklearn.tree import DecisionTreeClassifier


pl.set_params(est=DecisionTreeClassifier(random_state=3))

pl.fit(X_train, y_train)
y_pred = pl.predict(X_test)
print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.84      0.82      0.83       164
           1       0.72      0.75      0.74       104

    accuracy                           0.79       268
   macro avg       0.78      0.78      0.78       268
weighted avg       0.79      0.79      0.79       268



In [10]:
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transfomer example
class FamilyFeatures(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self  # No statistics to learn

    def transform(self, X):
        X = X.copy()  # Preserve the original DataFrame
        X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
        X["IsAlone"] = (X["FamilySize"] == 1).astype(int)
        return X